## Request Data from Coinalyse API
Documents: https://api.coinalyze.net/v1/doc/

### 1 - Import Libraries

In [1]:
import json as jn
import requests as rq
from datetime import datetime as dt
import pandas as pd

### 2 - Parametres pre-definition

In [2]:
# the base url
url = "https://api.coinalyze.net/v1/"

# list of data to call from api
url_espec = ["ohlcv-history", "long-short-ratio-history", "liquidation-history", "predicted-funding-rate-history",  "funding-rate-history", "open-interest-history"]

# the key to access the api
api_key = {"api_key" : "6fbe6e41-8f2b-4e30-a0fd-8c663beea2f4"}

# this correspond to the first moment of the bitcoin in the exchange in miliseconds
from_f = 1568260800

# now format in date time
to_t = int(dt.now().timestamp())

btc = 'BTCUSDT'

### 3 - Calling from API -- Bitcoin from Binance (BTC/USDT)

#### 3.1 - Exchanges

In [3]:
# call the api for the name and code for each exchange
exchanges = rq.get(url + 'exchanges', headers=api_key)
# saves in dataframe
exchanges_df = pd.DataFrame(data=exchanges.json())

In [4]:
exchanges_df.head()

,name,code
0,Poloniex,P
1,Vertex,V
2,Bitforex,D
3,Kraken,K
4,Bithumb,U


In [5]:
# retorn the code for binance ('A')
binance_code = exchanges_df[exchanges_df['name'] == 'Binance']['code'].iloc[0]

#### 3.2 - Markets

In [6]:
# call the api for the trade pairs
market = rq.get(url + 'future-markets', headers=api_key)
# saves in dataframe
market_df = pd.DataFrame(data=market.json())

In [7]:
market_df.head()

,symbol,exchange,symbol_on_exchange,base_asset,quote_asset,expire_at,has_buy_sell_data,is_perpetual,margined,oi_lq_vol_denominated_in,has_long_short_ratio_data,has_ohlcv_data
0,MNTUSDT.6,6,MNTUSDT,MNT,USDT,NaN,True,True,STABLE,BASE_ASSET,True,True
1,QNTUSDT.6,6,QNTUSDT,QNT,USDT,NaN,True,True,STABLE,BASE_ASSET,True,True
2,MAVUSDT.6,6,MAVUSDT,MAV,USDT,NaN,True,True,STABLE,BASE_ASSET,True,True
3,NMRUSDT_PERP.A,A,NMRUSDT,NMR,USDT,NaN,True,True,STABLE,BASE_ASSET,True,True
4,L3USDT.6,6,L3USDT,L3,USDT,NaN,True,True,STABLE,BASE_ASSET,True,True


In [8]:
# retorn the symbol for Bitcoin ('BTCUSDT_PERP.A')
btc_binance = market_df[(market_df['exchange'] == binance_code) & (market_df['symbol_on_exchange'] == btc)]['symbol'].iloc[0]

#### 3.3 - The Data

In [9]:
# the parametres that defines:
ohlc_params = {
            "symbols" : btc_binance, # the name of the pair (BTC/USD) futures from Binance
            "interval" : "daily", # the timeframe
            "from" : from_f, 
            "to" : to_t
            }

In [10]:
# the function to call the api
def data_dumper(endpoint):   
    response = rq.get(
                    f"{url}{endpoint}", 
                    params = ohlc_params, 
                    headers = api_key
                    )
    return response

In [11]:
url_espec[0]

'ohlcv-history'

In [12]:
# the dictionary where the data will be stored
data = {}

# a loop to call each individual data and put on the dictionary
for endpoint in url_espec:
    data[endpoint] = data_dumper(endpoint).json()

### 4 - Transform to a CSV file

In [13]:
# the function to put every call to a single dataframe
def df_maker(key, value, df_data):
    df = pd.DataFrame(data=value[0]['history'])
    
    # a loop to rename every column but 't'
    for c in df.columns:
        if c != 't':
            df.rename(columns={c: c + '_' + key.replace('-history', '')}, inplace=True)
        
    # return the dataframe if is the first time    
    if df_data is None:
        return df
    # merge if is not
    else:
        return pd.merge(df_data, df, how='left', on='t')      

In [14]:
# initiate the dataframe
df_data = None

# a loop to transform every data to a single dataframe
for key, value in data.items():
    df_data = df_maker(key, value, df_data) 

In [15]:
df_data.tail()

,t,o_ohlcv,h_ohlcv,l_ohlcv,c_ohlcv,v_ohlcv,bv_ohlcv,tx_ohlcv,btx_ohlcv,r_long-short-ratio,...,l_predicted-funding-rate,c_predicted-funding-rate,o_funding-rate,h_funding-rate,l_funding-rate,c_funding-rate,o_open-interest,h_open-interest,l_open-interest,c_open-interest
2219,1760054400,121579.4,122497.0,102648.4,112714.9,283074.793,132251.301,3255324,1602863.0,0.9040,...,-0.002106,0.010000,0.000785,0.003879,0.000785,0.001013,96722.011,97084.623,72364.700,72439.474
2220,1760140800,112715.0,113300.1,109501.0,110579.1,162986.706,73542.497,2041944,983670.0,1.1858,...,-0.002999,-0.002999,0.010000,0.010000,0.000467,0.000467,72434.743,76479.026,71421.781,74324.913
2221,1760227200,110579.2,115730.3,109509.3,114894.4,214563.112,107056.890,2305725,1149944.0,1.4021,...,-0.016207,-0.003878,-0.003055,0.002865,-0.010915,-0.010915,74331.358,76356.016,70503.009,75910.074
2222,1760313600,114894.5,115912.0,113600.5,115111.9,190121.618,92858.401,1858297,927903.0,1.5556,...,-0.005810,-0.004072,-0.003942,0.002519,-0.003942,-0.002305,75909.323,77513.040,75673.319,76664.481
2223,1760400000,115112.0,115360.2,109969.2,110665.2,145146.149,70047.343,1233227,616776.0,1.3299,...,-0.005017,-0.004820,-0.004041,0.000712,-0.004041,0.000712,76664.703,79261.712,75710.569,78994.003


In [16]:
df_data.columns

Index(['t', 'o_ohlcv', 'h_ohlcv', 'l_ohlcv', 'c_ohlcv', 'v_ohlcv', 'bv_ohlcv',
       'tx_ohlcv', 'btx_ohlcv', 'r_long-short-ratio', 'l_long-short-ratio',
       's_long-short-ratio', 'l_liquidation', 's_liquidation',
       'o_predicted-funding-rate', 'h_predicted-funding-rate',
       'l_predicted-funding-rate', 'c_predicted-funding-rate',
       'o_funding-rate', 'h_funding-rate', 'l_funding-rate', 'c_funding-rate',
       'o_open-interest', 'h_open-interest', 'l_open-interest',
       'c_open-interest'],
      dtype='object')

### 5 - Rename columns and save as CSV

In [17]:
# rename some columns manualy
rename_map = {
    'v_ohlcv': 'volume',
    'bv_ohlcv': 'buy_volume',
    'tx_ohlcv': 'transactions',
    'btx_ohlcv': 'buy_transactions'
}

df_data.rename(columns=rename_map, inplace=True)

new_cols = {}

# loop to rename some columns
for col in df_data.columns:
    new_col = col

    # timestamp
    if col == 't':
        new_col = 'datetime'

    # OHLCV prefixes
    elif col.startswith('o_'):
        new_col = col.replace('o_', 'open_')
    elif col.startswith('h_'):
        new_col = col.replace('h_', 'high_')
    elif col.startswith('l_'):
        new_col = col.replace('l_', 'low_')
    elif col.startswith('c_'):
        new_col = col.replace('c_', 'close_')

    # fix some indicator names
    if 'long-short-ratio' in new_col:
        new_col = new_col.replace('low_', 'long_')  
        new_col = new_col.replace('s_', 'short_') 
        new_col = new_col.replace('r_', '')         
    if 'liquidation' in new_col:
        new_col = new_col.replace('low_', 'long_')  
        new_col = new_col.replace('s_', 'short_')
    
    # chances ohlcv to price
    new_col = new_col.replace('ohlcv', 'price')

    # changes hifen to undescore
    new_col = new_col.replace('-', '_')

    new_cols[col] = new_col

df_data.rename(columns=new_cols, inplace=True)


In [18]:
df_data.columns

Index(['datetime', 'open_price', 'high_price', 'low_price', 'close_price',
       'volume', 'buy_volume', 'transactions', 'buy_transactions',
       'long_short_ratio', 'long_long_short_ratio', 'short_long_short_ratio',
       'long_liquidation', 'short_liquidation', 'open_predicted_funding_rate',
       'high_predicted_funding_rate', 'low_predicted_funding_rate',
       'close_predicted_funding_rate', 'open_funding_rate',
       'high_funding_rate', 'low_funding_rate', 'close_funding_rate',
       'open_open_interest', 'high_open_interest', 'low_open_interest',
       'close_open_interest'],
      dtype='object')

### 6 - Data clean plus add some new columns

#### 6.1 - Transforme `datetime` column to `year`, `month` `day` format.

In [19]:
df_data["datetime"] = pd.to_datetime(df_data["datetime"], unit="s")

#### 6.2- Take out some undesired columns

In [20]:
df_data.drop(columns=['long_long_short_ratio', 'short_long_short_ratio'], inplace=True)

#### 6.3- Add new columns

In [21]:
df_data['volume_delta'] = df_data.buy_volume - (df_data.volume - df_data.buy_volume)
df_data['cumulative_volume_delta'] = df_data.volume_delta.cumsum()
df_data['transaction_delta'] = df_data.buy_transactions - (df_data.transactions - df_data.buy_transactions)
df_data['cumulative_transaction_delta'] = df_data.transaction_delta.cumsum()
df_data['liquidation_delta'] = df_data.long_liquidation - df_data.short_liquidation
df_data['cumulative_liquidation_delta'] = df_data.liquidation_delta.cumsum()

In [22]:
# saves in CSV file    
df_data.to_csv("futures_raw_data.csv", index=False)  